
read the fact table

In [0]:
%python

df_silver = spark.sql("select * from parquet.`abfss://silver@pmngradls.dfs.core.windows.net/Clean Data/`")
df_silver.display()


read all the dimension table

In [0]:
dim_branch = spark.sql("select * from cars_catalog.gold.dim_branch")

dim_date = spark.sql("select * from cars_catalog.gold.dim_date")

dim_dealer = spark.sql("select * from cars_catalog.gold.dim_dealer")

dim_model = spark.sql("select * from cars_catalog.gold.dim_model")


create final fact table.

In [0]:
from pyspark.sql.functions import *

In [0]:
df_fact = df_silver.alias("silver").join(dim_branch.alias("branch"), col("silver.Branch_ID") == col("branch.branch_id"), 'left') \
                   .join(dim_date.alias("date"), col("silver.date_ID") ==  col("date.date_id"), 'left')\
                   .join(dim_dealer.alias("dealer"), col("silver.dealer_ID") == col("dealer.dealer_id"), 'left') \
                   .join(dim_model.alias("model"), col("silver.model_ID") ==  col("model.model_id"), 'left')\
                    .select(col("silver.Revenue"),col("silver.Units_Sold"),col("silver.Revenue_Per_Unit"),col("branch.dim_branch_key"),col("dealer.dim_dealer_key"),col("model.dim_model_key"),col("date.dim_date_key"))                

In [0]:
df_fact.display()

SCD type -1 upsert logic.

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("cars_catalog.gold.df_fact"):

    source_table = DeltaTable.forPath(spark, "abfss://gold@pmngradls.dfs.core.windows.net/df_fact/")

    source_table.alias("source").merge(
        df_fact.alias("target"),
        "source.dim_branch_key = target.dim_branch_key AND source.dim_dealer_key = target.dim_dealer_key AND source.dim_model_key = target.dim_model_key AND source.dim_date_key = target.dim_date_key"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll()\
    .execute()

else:
    df_fact = df_fact.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@pmngradls.dfs.core.windows.net/df_fact/")\
        .saveAsTable("cars_catalog.gold.df_fact")

In [0]:
%sql

select * from cars_catalog.gold.df_fact;